# 📊 Página 1 del BI — Indicadores de interés + Gráfica de Ejecución

Este notebook replica el procesamiento de la primera página del Power BI oficial del UBPD usando **solo pandas + numpy**.

Pasos:
1. Carga de datos desde **Excel** (`Metas 2026_GITT_ Para power BI.xlsx`)
2. Carga de datos desde **Base de Datos** (vía `GET /api/admin/bi/raw`)
3. Descripción y comparación de las dos fuentes — ambos deben dar los mismos resultados
4. Elección de la fuente activa (puedes cambiarla en una sola celda)
5. Limpieza y filtros por defecto (los mismos que ve un usuario al abrir la plataforma)
6. Agregación para las 13 tarjetas de **Indicadores de interés**
7. Gráfica de **Ejecución mensual 2026** (barras)
8. Gráfica de **Dato 2025** (línea horizontal de referencia)

El resultado debe coincidir 1-a-1 con lo que se visualiza en el BI oficial.

## 1. Imports

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# helpers de carga (Excel vs DB) — ver src/loaders.py
sys.path.insert(0, str(Path.cwd() / "src"))
from loaders import load_from_excel, load_from_db, describe_dict, compare_sources

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 180)

## 2. Cargar desde el Excel

Construimos un diccionario donde **cada key es el nombre de la hoja** y el valor es el `DataFrame` correspondiente.

In [ ]:
excel_path = Path("data/Metas 2026_GITT_ Para power BI.xlsx")
data_excel = load_from_excel(excel_path)
print("Hojas cargadas desde Excel:", list(data_excel.keys()))

## 3. Cargar desde la base de datos

Se hace login como admin y se consume el endpoint `GET /api/admin/bi/raw`, que devuelve las mismas 3 hojas con los mismos nombres de columna del Excel. Esto permite contrastar que lo que ve el BI viene exactamente del mismo dataset.

In [ ]:
# Si tu API está en otro puerto/URL, ajústalo aquí
API_BASE   = "http://localhost/api"
USERNAME   = "admin"
PASSWORD   = "Admin@UBPD2024!"

try:
    data_db = load_from_db(API_BASE, USERNAME, PASSWORD)
    print("Hojas cargadas desde DB:", list(data_db.keys()))
except Exception as exc:
    print(f"⚠️  No se pudo conectar a la API: {exc}")
    data_db = None

## 4. Descripción comparativa de ambas fuentes

Imprimimos forma, columnas y tipos de cada DataFrame de ambas fuentes.

In [ ]:
describe_dict(data_excel, titulo="📄 Fuente 1: Excel")
if data_db is not None:
    describe_dict(data_db, titulo="🗄️  Fuente 2: Base de Datos")

In [ ]:
# Tabla comparativa de conteos
if data_db is not None:
    compare_sources(data_excel, data_db)

> **Nota:** El Excel tiene más filas en `Historico` (1703 vs 1248 en DB). La diferencia son 455 filas con `AÑO` vacío (PRBs "Sin Determinar") que el BI filtra automáticamente. Lo replicaremos en el paso siguiente.

## 5. Seleccionar fuente activa

Cambia `SOURCE` a `"excel"` o `"db"` para usar uno u otro. **Todo el resto del notebook corre con la variable `data`**, así que cambiar la fuente en esta celda es suficiente.

In [ ]:
SOURCE = "excel"   # "excel" | "db"

if SOURCE == "excel":
    data = data_excel
elif SOURCE == "db":
    assert data_db is not None, "data_db no disponible"
    data = data_db
else:
    raise ValueError(f"SOURCE inválido: {SOURCE}")

print(f"🔵 Fuente activa: {SOURCE}")
print(f"   Historico: {data['Historico'].shape}")

## 6. Vista rápida de las 3 tablas

In [ ]:
data["PRB"].head()

In [ ]:
data["EstructuraIndicadores"][["Cod_Indicador", "Código del Indicador", "Indicador"]]

In [ ]:
data["Historico"].head(3)

## 7. Limpieza — replicar el filtro implícito del BI

El BI aplica por defecto `AÑO = 2026` y descarta filas donde `CodPRB` o `Cod_Indicador` estén vacíos. Lo hacemos explícitamente:

In [ ]:
hist = data["Historico"].copy()
antes = len(hist)

hist = hist.dropna(subset=["CodPRB", "Cod_Indicador", "AÑO"])
hist = hist[(hist["CodPRB"] != 0) & (hist["Cod_Indicador"] != 0)]

# Tipos numéricos
for col in ["Meta", "Avance Total", "Línea Base 2025"] + [f"Mes {m}" for m in range(1, 13)]:
    if col in hist.columns:
        hist[col] = pd.to_numeric(hist[col], errors="coerce").fillna(0)

print(f"Filas antes:   {antes}")
print(f"Filas después: {len(hist)}")
print(f"Años únicos:   {sorted(hist['AÑO'].dropna().unique().astype(int).tolist())}")

## 8. Valores únicos → selectores de la plataforma

Extraemos los valores que pueblan los selectores del dashboard.

In [ ]:
prb_df = data["PRB"].dropna(subset=["COD"]).copy()
est_df = data["EstructuraIndicadores"].copy()

anios_opts       = sorted(hist["AÑO"].dropna().astype(int).unique().tolist())
regionales_opts  = sorted(prb_df["Regional"].dropna().unique().tolist())
gitts_opts       = sorted(prb_df["GITT"].dropna().unique().tolist())
prb_opts         = sorted(prb_df["PRB"].dropna().unique().tolist())
indicador_opts   = est_df[["Código del Indicador", "Indicador"]].to_dict(orient="records")

print(f"Años:         {anios_opts}")
print(f"Regionales:   {regionales_opts}")
print(f"GITTs:        {len(gitts_opts)} ({gitts_opts[:5]}…)")
print(f"PRBs:         {len(prb_opts)} ({prb_opts[:3]}…)")
print(f"Indicadores:  {len(indicador_opts)}")

## 9. Aplicar filtros por DEFECTO

Los mismos que están seleccionados cuando abres el BI:

| Filtro | Valor por defecto |
|---|---|
| Año | 2026 |
| Trimestre / Mes | Todos |
| Regional | Todas |
| GITT | Todos |
| PRB | Todos |
| Nombre del indicador | Todos |

In [ ]:
# Filtros activos (cambialos si quieres reproducir otro escenario)
F_ANIO        = 2026
F_REGIONAL    = None    # None = Todas
F_GITT        = None
F_PRB         = None
F_INDICADOR   = None

f = hist.copy()
if F_ANIO is not None:
    f = f[f["AÑO"] == F_ANIO]

# Filtros que requieren join con PRB
if F_REGIONAL or F_GITT or F_PRB:
    f = f.merge(prb_df[["COD", "Regional", "GITT", "PRB"]].rename(columns={"PRB": "PRB_nombre"}),
                left_on="CodPRB", right_on="COD", how="left")
    if F_REGIONAL: f = f[f["Regional"] == F_REGIONAL]
    if F_GITT:     f = f[f["GITT"] == F_GITT]
    if F_PRB:      f = f[f["PRB_nombre"] == F_PRB]

if F_INDICADOR:
    f = f[f["Código del Indicador"] == F_INDICADOR]

print(f"Filas tras filtros: {len(f)}")

## 10. Agregación — 13 tarjetas de Indicadores de interés

Por cada `Código del Indicador` calculamos:
- **Dato 2025** = Σ `Línea Base 2025`
- **Avance 2026** = Σ `Avance Total`
- **Meta 2026** = Σ `Meta`
- **% Avance** = Avance / Meta × 100

In [ ]:
tarjetas = (
    f.groupby(["Cod_Indicador", "Código del Indicador", "Indicador"], as_index=False)
      .agg(
          dato_2025   = ("Línea Base 2025", "sum"),
          avance_2026 = ("Avance Total",    "sum"),
          meta_2026   = ("Meta",            "sum"),
      )
)
tarjetas["pct_avance"] = np.where(
    tarjetas["meta_2026"] > 0,
    tarjetas["avance_2026"] / tarjetas["meta_2026"] * 100,
    0,
)
tarjetas = tarjetas.sort_values("Cod_Indicador").reset_index(drop=True)
tarjetas

## 11. Visualización de las tarjetas como grid

Mini-cards con el mismo formato del BI.

In [ ]:
n = len(tarjetas)
cols = 4
rows = (n + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(16, rows * 2.4))
axes = axes.flatten()

PURPLE = "#5B2B5E"

for i, t in tarjetas.iterrows():
    ax = axes[i]
    ax.axis("off")
    titulo = t["Indicador"][:55]
    ax.set_title(titulo, fontsize=9, fontweight="bold", pad=6)
    # 3 números
    ax.text(0.00, 0.60, f"{t['dato_2025']:,.0f}",   fontsize=12, fontweight="bold")
    ax.text(0.00, 0.48, "Dato 2025",                fontsize=7, color="gray", style="italic")
    ax.text(0.38, 0.60, f"{t['avance_2026']:,.0f}", fontsize=12, fontweight="bold")
    ax.text(0.38, 0.48, "Avance 2026",              fontsize=7, color="gray", style="italic")
    ax.text(0.76, 0.60, f"{t['meta_2026']:,.0f}",   fontsize=12, fontweight="bold")
    ax.text(0.76, 0.48, "Meta 2026",                fontsize=7, color="gray", style="italic")
    # barra de avance
    pct = min(100, t["pct_avance"]) / 100
    ax.add_patch(plt.Rectangle((0, 0.32), 1,   0.04, color="#E5E7EB"))
    ax.add_patch(plt.Rectangle((0, 0.32), pct, 0.04, color=PURPLE))
    ax.text(0.5, 0.08, f"{t['pct_avance']:.1f} %", fontsize=14, fontweight="bold",
            ha="center", color=PURPLE)

for j in range(n, len(axes)): axes[j].axis("off")
plt.suptitle("Indicadores de interés — Página 1 del BI", fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

## 12. Gráfica de barras — Ejecución mensual 2026

Suma de `Mes 1` a `Mes 12` para todas las filas filtradas.

In [ ]:
MES_COLS    = [f"Mes {m}" for m in range(1, 13)]
MES_NOMBRES = ["January", "February", "March", "April", "May", "June",
               "July", "August", "September", "October", "November", "December"]

ejecucion_mensual = f[MES_COLS].sum().values
ejecucion_mensual

In [ ]:
fig, ax = plt.subplots(figsize=(13, 4.2))

bars = ax.bar(MES_NOMBRES, ejecucion_mensual, color="#D1D5DB", width=0.6, label="Ejecución 2026")

# etiqueta sobre cada barra
for bar, v in zip(bars, ejecucion_mensual):
    if v > 0:
        ax.text(bar.get_x() + bar.get_width()/2, v, f"{v:,.0f}",
                ha="center", va="bottom", fontsize=9, color="#444")

ax.set_ylabel("Ejecución 2026")
ax.set_xlabel("Mes")
ax.set_title("Ejecución mensual — 2026", fontsize=12, fontweight="bold")
ax.spines[["top", "right"]].set_visible(False)
ax.tick_params(axis="x", labelrotation=30, labelsize=9)
plt.tight_layout()
plt.show()

## 13. Gráfica de línea — Dato 2025 (referencia)

Es una constante horizontal = Σ `Línea Base 2025` con los filtros actuales. En el BI aparece sobre la gráfica de barras.

In [ ]:
dato_2025 = float(f["Línea Base 2025"].sum())
print(f"Dato 2025 = {dato_2025:,.0f}")

In [ ]:
fig, ax = plt.subplots(figsize=(13, 2.6))
ax.plot(MES_NOMBRES, [dato_2025] * 12, color="#A43F5F", linewidth=2,
        marker="o", markersize=5, label="Dato 2025")
for i, n in enumerate(MES_NOMBRES):
    ax.annotate(f"{dato_2025:,.0f}", (n, dato_2025), textcoords="offset points",
                xytext=(0, 8), ha="center", fontsize=8, color="#A43F5F")
ax.set_title("Dato 2025 — línea de referencia", fontsize=12, fontweight="bold")
ax.spines[["top", "right", "left"]].set_visible(False)
ax.set_yticks([])
ax.tick_params(axis="x", labelrotation=30, labelsize=9)
plt.tight_layout()
plt.show()

## 14. Gráfica combinada (como aparece en el BI)

Barras de Ejecución 2026 + línea de Dato 2025 en el mismo plot.

In [ ]:
fig, ax = plt.subplots(figsize=(13, 5))

# Barras — ejecución mensual
bars = ax.bar(MES_NOMBRES, ejecucion_mensual, color="#D1D5DB", width=0.55, label="Ejecución 2026")
for bar, v in zip(bars, ejecucion_mensual):
    if v > 0:
        ax.text(bar.get_x() + bar.get_width()/2, v, f"{v:,.0f}",
                ha="center", va="bottom", fontsize=9, color="#444")

# Línea — dato 2025
ax.plot(MES_NOMBRES, [dato_2025] * 12, color="#A43F5F", linewidth=2,
        marker="o", markersize=5, label="Dato 2025")

ax.set_ylabel("Ejecución 2026")
ax.set_xlabel("Mes")
ax.set_title("Ejecución mensual vs Dato 2025 — Página 1 del BI",
             fontsize=13, fontweight="bold")
ax.legend(loc="upper right", frameon=False)
ax.spines[["top", "right"]].set_visible(False)
ax.tick_params(axis="x", labelrotation=30, labelsize=9)
plt.tight_layout()
plt.show()

## 15. Validación final

Resumen de KPIs globales que se pueden contrastar contra el BI.

In [ ]:
resumen = pd.Series({
    "Total Meta":            f["Meta"].sum(),
    "Total Avance":          f["Avance Total"].sum(),
    "Total Dato 2025":       f["Línea Base 2025"].sum(),
    "% Avance global":       f["Avance Total"].sum() / f["Meta"].sum() * 100 if f["Meta"].sum() else 0,
    "PRBs únicos":           f["CodPRB"].nunique(),
    "Indicadores únicos":    f["Cod_Indicador"].nunique(),
    "Filas procesadas":      len(f),
})
resumen.round(2)

✅ **Estos valores deben coincidir con los que muestra el BI oficial** cuando se abre con los filtros por defecto (AÑO=2026, todo lo demás sin filtrar).

Para validar también contra la implementación backend del dashboard UBPD puedes llamar al endpoint público:

```
GET /api/bi/kpis?anio=2026
```

y comparar.